In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
%pip install "pillow>=10.0,<13.0"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import transformers
import PIL

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pillow:", PIL.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
    

PyTorch: 2.11.0+cu128
Transformers: 5.22.1
Pillow: 13.0.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [8]:
%pip install -q -U transformers accelerate huggingface_hub
%pip install -q fastapi uvicorn pyngrok
    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch

print("BF16 supported:", torch.cuda.is_bf16_supported())
print("GPU:", torch.cuda.get_device_name(0))

BF16 supported: True
GPU: Tesla T4


In [14]:
!pip install -U transformers

In [15]:
from huggingface_hub import login
login()

In [16]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
)

model.eval()

print("Model loaded:", MODEL_ID)
print("Precision:", dtype)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

... (6 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model loaded: google/medgemma-4b-it
Precision: torch.bfloat16
GPU memory allocated: 5.37 GiB


In [17]:
import torch

print("GPU podržava BF16:", torch.cuda.is_bf16_supported())
print("Model device:", model.device)
print("Model dtype:", next(model.parameters()).dtype)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Explain the difference between iron deficiency "
                    "and vitamin B12 deficiency in three short bullet points."
                ),
            }
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

# Ispisujemo samo novogenerirani odgovor, bez ulaznog upita.
new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = processor.decode(
    new_tokens,
    skip_special_tokens=True,
)

print("\n--- MEDGEMMA ODGOVOR ---\n")
print(answer)

GPU podržava BF16: True
Model device: cuda:0
Model dtype: torch.bfloat16

--- MEDGEMMA ODGOVOR ---

Here's the difference between iron and vitamin B12 deficiency in three bullet points:

*   **Iron Deficiency:** Primarily affects red blood cell production, leading to anemia (low red blood cell count) and fatigue. It's often caused by inadequate iron intake, poor absorption, or blood loss.

*   **Vitamin B12 Deficiency:** Impacts nerve function and red blood cell production, leading to anemia, neurological problems (like numbness, tingling, and memory loss), and fatigue. It's often caused by poor absorption due to age, certain medical conditions, or dietary restrictions.

*   **Key Difference:** Iron deficiency primarily affects red blood cells, while vitamin B12 deficiency affects both red blood cells and nerve function.



In [18]:
%pip install -q fastapi uvicorn pyngrok


Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
import transformers
import importlib.util

print("CUDA dostupna:", torch.cuda.is_available())
print("FastAPI instaliran:", importlib.util.find_spec("fastapi") is not None)
print("Uvicorn instaliran:", importlib.util.find_spec("uvicorn") is not None)
print("Pyngrok instaliran:", importlib.util.find_spec("pyngrok") is not None)
print("Model u memoriji:", "model" in globals())
print("Processor u memoriji:", "processor" in globals())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA dostupna: True
FastAPI instaliran: True
Uvicorn instaliran: True
Pyngrok instaliran: True
Model u memoriji: True
Processor u memoriji: True
GPU: Tesla T4


In [20]:
import os
import secrets
import threading
import time

import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

# Generiraj privatni API ključ.
# Sačuvaj ga jer će ti trebati za pozive iz drugih aplikacija.
if "MEDGEMMA_API_KEY" not in os.environ:
    os.environ["MEDGEMMA_API_KEY"] = secrets.token_urlsafe(32)

API_KEY = os.environ["MEDGEMMA_API_KEY"]

app = FastAPI(
    title="Private MedGemma API",
    version="1.0.0",
)

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=12000)
    max_new_tokens: int = Field(default=512, ge=1, le=1024)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "google/medgemma-4b-it",
        "gpu": torch.cuda.get_device_name(0),
    }

@app.post("/generate")
def generate(
    request: GenerateRequest,
    x_api_key: str = Header(default=""),
):
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": request.prompt,
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            do_sample=False,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = processor.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    return {
        "model": "google/medgemma-4b-it",
        "response": answer,
    }

def run_api():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="warning",
    )

# Pokreni server u pozadinskoj niti.
if "api_thread" not in globals() or not api_thread.is_alive():
    api_thread = threading.Thread(target=run_api, daemon=True)
    api_thread.start()
    time.sleep(2)

print("API ključ:", API_KEY)
print("Lokalni API: http://127.0.0.1:8000")
print("API dokumentacija: http://127.0.0.1:8000/docs")

API ključ: SeLKPwBve62T27wv33gvtHPr0J2JB3p4X-kwQ26dHt4
Lokalni API: http://127.0.0.1:8000
API dokumentacija: http://127.0.0.1:8000/docs


In [ ]:
!pip install requests

In [26]:
import requests
print("Requests je spreman:", requests.__version__)

Requests je spreman: 2.34.2


In [41]:
import secrets

API_KEY = secrets.token_urlsafe(32)
print("Novi API ključ generiran.")

Novi API ključ generiran.


In [31]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/generate",
    headers={"X-API-Key": API_KEY},
    json={
        "prompt": "Explain the difference between iron deficiency and vitamin B12 deficiency in three bullet points.",
        "max_new_tokens": 200,
    },
    timeout=180,
)

print("HTTP status:", response.status_code)
print(response.json())

HTTP status: 200
{'model': 'google/medgemma-4b-it', 'response': "Here's a breakdown of the key differences between iron and vitamin B12 deficiencies:\n\n*   **Cause:**\n    *   **Iron Deficiency:** Primarily caused by inadequate dietary intake, poor absorption, or blood loss (e.g., menstruation, gastrointestinal bleeding).\n    *   **Vitamin B12 Deficiency:** Primarily caused by inadequate dietary intake (especially in vegans/vegetarians), impaired absorption (e.g., pernicious anemia, atrophic gastritis), or certain medications.\n\n*   **Primary Symptoms:**\n    *   **Iron Deficiency:** Fatigue, weakness, pale skin, shortness of breath, headaches, dizziness, brittle nails, and pica (craving non-food items).\n    *   **Vitamin B12 Deficiency:** Fatigue, weakness, numbness or tingling in hands and feet (peripheral neuropathy), difficulty walking, memory problems, depression, and glossitis (inflamed tongue).\n\n*   **Long-Term Consequences:**\n    *   **Iron"}


In [3]:
!pip install pyngrok


In [5]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

# 1. Učitaj token iz Kaggle Secrets
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")

# 2. Postavi autentifikaciju
ngrok.set_auth_token(ngrok_token)

# 3. Provjeri postojeće tunele
tunnels = ngrok.get_tunnels()

# 4. Otvori tunel samo ako već ne postoji
if not any("8000" in str(t.config.get("addr")) for t in tunnels):
    tunnel = ngrok.connect(8000, "http")

# 5. Ispiši aktivne javne adrese
print("Aktivni tuneli:")
for tunnel in ngrok.get_tunnels():
    print(tunnel.public_url)

Aktivni tuneli:                                                                                     
https://37c5-34-91-1-142.ngrok-free.app


t=2026-11-01T15:15:45+0000 lvl=warn msg="failed to open private leg" id=9852840a5773 privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


In [39]:

@app.post("/generate")
async def generate(request: GenerateRequest, x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "Invalid API key")

In [42]:
print("my: ", API_KEY)

my:  bP2nnrOKRWa_cZvTHbCcJ_wRZR0g4ZbHi9W8j7ZDOCw


In [6]:
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional
import os

app = FastAPI(title="Private MedGemma API", version="1.0.0")

# Your API key (or from Secrets)
API_KEY = os.environ.get("MEDGEMMA_API_KEY", "your-secret-key-here")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

class GenerateResponse(BaseModel):
    response: str

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "google/medgemma-4b-it",
        "gpu": "Tesla T4"
    }

@app.post("/generate", response_model=GenerateResponse)
def generate(request: GenerateRequest, x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "Invalid API key")
    
    # YOUR MODEL INFERENCE CODE HERE
    # e.g., response = model.generate(request.prompt, max_new_tokens=request.max_new_tokens)
    response = f"Model response to: {request.prompt}"  # placeholder
    
    return GenerateResponse(response=response)

# Run the server
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [15]:
# Cell 2: Apply nest_asyncio (MUST run before uvicorn)
import nest_asyncio
nest_asyncio.apply()

In [17]:
# Cell 3: Define and run the API
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import os

app = FastAPI(title="Private MedGemma API", version="1.0.0")

API_KEY = os.environ.get("MEDGEMMA_API_KEY", "your-secret-key-here")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

class GenerateResponse(BaseModel):
    response: str

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "google/medgemma-4b-it",
        "gpu": "Tesla T4"
    }

@app.post("/generate", response_model=GenerateResponse)
def generate(request: GenerateRequest, x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "Invalid API key")
    
    # YOUR ACTUAL MODEL CODE HERE
    # response = model.generate(request.prompt, max_new_tokens=request.max_new_tokens)
    response = f"Model response to: {request.prompt}"  # Replace with real inference
    
    return GenerateResponse(response=response)

# Cell 4: Run server (blocks - keep this cell running)
uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [18]:
%pip install fastapi uvicorn pydantic nest-asyncio -q


Note: you may need to restart the kernel to use updated packages.


In [19]:
import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio applied")


nest_asyncio applied


In [20]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import os

app = FastAPI(title="Private MedGemma API", version="1.0.0")

API_KEY = os.environ.get("MEDGEMMA_API_KEY", "your-secret-key-here")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

class GenerateResponse(BaseModel):
    response: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "google/medgemma-4b-it", "gpu": "Tesla T4"}

@app.post("/generate", response_model=GenerateResponse)
def generate(request: GenerateRequest, x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "Invalid API key")
    
    # YOUR MODEL CODE HERE
    response = f"Model response to: {request.prompt}"
    
    return GenerateResponse(response=response)

print("App defined")

App defined


In [5]:
# Kill any process on port 8000
import subprocess
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
print("Port 8000 cleared")


Port 8000 cleared


In [6]:
# Complete setup in ONE cell - run this alone
%pip install fastapi uvicorn pydantic nest-asyncio -q

import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import os
import uvicorn
import threading

app = FastAPI(title="Private MedGemma API", version="1.0.0")

API_KEY = os.environ.get("MEDGEMMA_API_KEY", "your-secret-key-here")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

class GenerateResponse(BaseModel):
    response: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "google/medgemma-4b-it", "gpu": "Tesla T4"}

@app.post("/generate", response_model=GenerateResponse)
def generate(request: GenerateRequest, x_api_key: str = Header(...)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "Invalid API key")
    
    # REPLACE WITH YOUR ACTUAL MODEL CODE
    response = f"Model response to: {request.prompt}"
    
    return GenerateResponse(response=response)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("✅ Server started in background on http://0.0.0.0:8000")
print("Test: curl https://37c5-34-91-1-142.ngrok-free.app/health")

Note: you may need to restart the kernel to use updated packages.
✅ Server started in background on http://0.0.0.0:8000
Test: curl https://37c5-34-91-1-142.ngrok-free.app/health
